# Tahlili AI server (Colab)
**قبل ما تشغل:** Runtime > Change runtime type > **T4 GPU**. وفي Secrets (أيقونة المفتاح) فعّل "Notebook access" وضيف:

| Secret | مطلوب؟ | إيه هو |
|---|---|---|
| `API_KEY` | موصى به | أي كلمة سر، والـ backend بيبعتها في الهيدر `X-API-KEY` |
| `GITHUB_TOKEN` | اختياري | لو الريبو private |

التونل بـ Cloudflare (من غير حساب ولا توكن). الرابط بيتغير كل مرة تشغّل فيها خلية 5 من الأول.

## 1. Config

In [18]:
import os, subprocess, json, time, sys, threading, re
from google.colab import userdata

def secret(k, default=""):
    try:
        return userdata.get(k) or default
    except Exception:
        return default

os.environ["API_KEY"] = secret("API_KEY", "dev-key")
os.environ["MODEL"] = "gemma4:e4b"       # جرّب "gemma4:26b" بعدين لو الذاكرة تكفي
GITHUB_TOKEN = secret("GITHUB_TOKEN")
REPO_URL = "https://github.com/amr305/ABC-Hachtoberfest-.git"
MODEL = os.environ["MODEL"]

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU - change runtime to T4")

GPU 0: Tesla T4 (UUID: GPU-0e9b65e0-0923-1177-f6e6-8aa788a59a32)



## 2. Install (one time per session, ~2-3 min)

In [19]:
!pip -q install fastapi uvicorn python-multipart requests pillow
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
!/content/cloudflared --version

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
cloudflared version 2026.10.4 (built 2026-10-26-11:12 UTC)


## 3. Start Ollama and pull Gemma 4

In [20]:
import requests
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(40):
    try:
        requests.get("http://localhost:11434", timeout=1); break
    except Exception:
        time.sleep(1)
!ollama pull {MODEL}
!ollama list


NAME          ID              SIZE      MODIFIED               
gemma4:e4b    dc35e8d9c606    6.6 GB    Less than a second ago    


## 4. Get the code (from GitHub, or upload `ai.zip`)

> Add blockquote



In [21]:
if REPO_URL:
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", url, "/content/repo"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/repo", "pull", "--rebase"], check=True)
    AI_DIR = "/content/repo/ai"
else:
    if not os.path.exists("/content/ai"):
        from google.colab import files
        files.upload()                       # choose ai.zip
        subprocess.run(["unzip", "-oq", "ai.zip", "-d", "/content"], check=True)
    AI_DIR = "/content/ai"
print("code at:", AI_DIR, os.listdir(AI_DIR))

code at: /content/repo/ai ['__pycache__', 'sample_explain_input.json', 'colab_server.py', 'prompts']


## 5. Start the server + public tunnel (re-run this cell after editing `colab_server.py`)

In [22]:
import importlib, uvicorn

if AI_DIR not in sys.path:
    sys.path.insert(0, AI_DIR)
import colab_server
importlib.reload(colab_server)

# وقّف السيرفر القديم لو الخلية اتشغلت قبل كده
try:
    server.should_exit = True
    time.sleep(1.5)
except NameError:
    pass

config = uvicorn.Config(colab_server.app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)

# تونل Cloudflare (بيتعمل مرة واحدة ويتعاد استخدامه)
if "cf_proc" not in globals() or cf_proc.poll() is not None:
    cf_proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
        stdout=open("/content/cf.log", "w"), stderr=subprocess.STDOUT)
    public_url = None
    for _ in range(40):
        time.sleep(1)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cf.log").read())
        if m:
            public_url = m.group(0)
            break

print("\nCOLAB_URL =", public_url)
print("API_KEY   =", os.environ["API_KEY"])
print("local health:", requests.get("http://localhost:8000/health").json())


COLAB_URL = https://cnet-compete-boot-justin.trycloudflare.com
API_KEY   = mohamedbehelkahtany
local health: {'ok': True, 'model': 'gemma4:e4b', 'model_loaded': True}


## 6. Test `/extract` with a lab sheet image (upload one)

In [23]:
from google.colab import files
up = files.upload()
name = list(up)[0]
t0 = time.time()
r = requests.post("http://localhost:8000/extract",
                  headers={"X-API-KEY": os.environ["API_KEY"]},
                  files={"file": (name, up[name])})
print(r.status_code, f"{time.time()-t0:.1f}s")
print(json.dumps(r.json(), ensure_ascii=False, indent=2))

Saving Screenshot 2026-10-26 175640.png to Screenshot 2026-10-26 175640.png
200 36.1s
{
  "report_type": "lab",
  "tests": [
    {
      "name": "Hemoglobin (Hb)",
      "value": 14.2,
      "value_text": null,
      "unit": "g/dL",
      "ref_low": 13.5,
      "ref_high": 17.5,
      "ref_text": "13.5 - 17.5"
    },
    {
      "name": "WBC (White Blood Cells)",
      "value": 6.8,
      "value_text": null,
      "unit": "10^3/uL",
      "ref_low": 4.0,
      "ref_high": 11.0,
      "ref_text": "4.0 - 11.0"
    },
    {
      "name": "Platelets",
      "value": 245,
      "value_text": null,
      "unit": "10^3/uL",
      "ref_low": 150,
      "ref_high": 450,
      "ref_text": "150 - 450"
    },
    {
      "name": "Fasting Blood Sugar",
      "value": 95,
      "value_text": null,
      "unit": "mg/dL",
      "ref_low": 70,
      "ref_high": 100,
      "ref_text": "70 - 100"
    },
    {
      "name": "Serum Creatinine",
      "value": 0.9,
      "value_text": null,
      "unit": "m

In [24]:
import base64, json, time, requests
from PIL import Image

SRC = "/content/xray.jpg"  # غيّر المسار لصورة الأشعة
im = Image.open(SRC).convert("RGB"); im.thumbnail((1400, 1400)); im.save("/content/xray_small.jpg", quality=85)
b64 = base64.b64encode(open("/content/xray_small.jpg", "rb").read()).decode()

PROMPT = """You are assisting a patient-education app. Look at this medical image.
Return ONLY JSON:
{"is_medical_image": true/false,
 "modality": "X-ray|CT|MRI|ultrasound|other|unknown",
 "body_part": "...",
 "image_quality": "good|fair|poor",
 "general_observations": ["up to 3 short, hedged, non-diagnostic observations, e.g. 'bone outlines appear visible'"],
 "arabic_summary": "جملتين بالعامية المصرية بتوصف نوع الصورة والعضو بس"}
Rules: do NOT diagnose. Do NOT say a fracture, tumor, or disease is present or absent. Do NOT give coordinates or locate abnormalities. If unsure, say unknown."""

t0 = time.time()
r = requests.post("http://localhost:11434/api/chat", json={
    "model": "gemma4:e4b", "stream": False, "format": "json",
    "messages": [{"role": "user", "content": PROMPT, "images": [b64]}]
}, timeout=300)
print(round(time.time() - t0, 1), "s")
print(json.dumps(json.loads(r.json()["message"]["content"]), ensure_ascii=False, indent=2))

33.6 s
{
  "is_medical_image": true,
  "modality": "other",
  "body_part": "unknown",
  "image_quality": "good",
  "general_observations": [
    "The document is a laboratory blood test report.",
    "Multiple hematological and biochemical parameters are listed.",
    "The results are provided with reference ranges."
  ],
  "arabic_summary": "دي ورقة بتحتوي على تحاليل دم للمريض. بتوريك قياسات مختلفة عشان تحدد الحالة الصحية العامة للمريض."
}


## 7. Test `/explain` with the sample input

In [25]:
sample = json.load(open(f"{AI_DIR}/sample_explain_input.json", encoding="utf-8"))
t0 = time.time()
r = requests.post("http://localhost:8000/explain",
                  headers={"X-API-KEY": os.environ["API_KEY"]}, json=sample)
print(r.status_code, f"{time.time()-t0:.1f}s")
print(json.dumps(r.json(), ensure_ascii=False, indent=2))

200 41.3s
{
  "summary": [
    "التحاليل دي بتوضح إن فيه انخفاض في مستويات الهيموجلوبين والحديد في جسمك.",
    "القيم دي محتاجة متابعة مع الدكتور عشان نعرف السبب ونشوف الخطوات الجاية.",
    "من فضلك استشير الدكتور اللي طلب التحاليل عشان يحدد الإجراء المناسب."
  ],
  "explanations": [
    {
      "name": "Hemoglobin",
      "text": "الهيموجلوبين ده بيقيس كمية الأكسجين اللي بيتنقل في الدم. القيمة المنخفضة ممكن تكون مرتبطة بفقر الدم (الأنيميا)."
    },
    {
      "name": "Ferritin",
      "text": "الفيريتين بيقيس مخزون الحديد في الجسم. القيمة المنخفضة ممكن تكون مرتبطة بنقص الحديد."
    },
    {
      "name": "HbA1c",
      "text": "ده بيقيس متوسط سكر الدم في آخر 3 شهور، والقيمة دي طبيعية وممتازة."
    },
    {
      "name": "TSH",
      "text": "TSH بيقيس وظائف الغدة الدرقية، والقيمة دي طبيعية ومستقرة."
    }
  ],
  "pattern": "الشكل ده من انخفاض الهيموجلوبين والفيريتين مع بعض ممكن يكون مرتبط بنقص الحديد في الجسم.",
  "priority": "follow_up",
  "doctor_questions": [
    "إيه الأسباب المح

## 8. Batch eval: run every image you upload and save the outputs (use for your accuracy check)

In [26]:
files_up = files.upload()
os.makedirs("/content/eval_out", exist_ok=True)
for fname, data in files_up.items():
    t0 = time.time()
    r = requests.post("http://localhost:8000/extract",
                      headers={"X-API-KEY": os.environ["API_KEY"]}, files={"file": (fname, data)})
    out = r.json()
    json.dump(out, open(f"/content/eval_out/{fname}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    n = len(out.get("tests", [])) if isinstance(out, dict) else "?"
    print(f"{fname}: {r.status_code}  {time.time()-t0:.1f}s  tests found: {n}")
print("saved in /content/eval_out")

saved in /content/eval_out


## 9. Push your changes to GitHub (only if you use REPO_URL)

In [27]:
msg = "ai: update"
subprocess.run("git config user.email 'colab@tahlili' && git config user.name 'colab'", shell=True, cwd="/content/repo")
print(subprocess.run(f"git add ai contract && git commit -m '{msg}' ; git pull --rebase && git push",
                     shell=True, cwd="/content/repo", capture_output=True, text=True))

CompletedProcess(args="git add ai contract && git commit -m 'ai: update' ; git pull --rebase && git push", returncode=1, stdout='On branch main\nYour branch is ahead of \'origin/main\' by 2 commits.\n  (use "git push" to publish your local commits)\n\nUntracked files:\n  (use "git add <file>..." to include in what will be committed)\n\tScreenshot 2026-10-26 175640.png\n\nnothing added to commit but untracked files present (use "git add" to track)\n', stderr='From https://github.com/amr305/ABC-Hachtoberfest-\n   f932425..52867c5  main       -> origin/main\nRebasing (1/2)\nerror: The following untracked working tree files would be overwritten by merge:\n\tai/__pycache__/colab_server.cpython-313.pyc\nPlease move or remove them before you merge.\nAborting\nhint: Could not execute the todo command\nhint: \nhint:     pick 66266d5bc199ac3cbeec50810edcc5a96bf54491 ai: update\nhint: \nhint: It has been rescheduled; To edit the command before continuing, please\nhint: edit the todo list first:\n

## 10. Keep-alive (run last and leave running while the team works)

In [ ]:
while True:
    time.sleep(60)